# 02 - Préprocessing (nettoyage + pondération)
Ce notebook nettoie `events.csv`, applique une pondération (view/cart/transaction) puis exporte un dataset prêt pour l'ALS.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

spark = SparkSession.builder.appName('ShopAI-02-Preprocessing').getOrCreate()

events_path = 'data/raw/events.csv'

events = spark.read.csv(events_path, header=True, inferSchema=True)
events.printSchema()


## 1) Nettoyage des données
- Supprimer les lignes nulles sur (visitorid, itemid, event)
- Supprimer les doublons (visitorid, itemid, event)


In [ ]:
before = events.count()

events_clean = events.dropna(subset=['visitorid','itemid','event'])
events_clean = events_clean.dropDuplicates(['visitorid','itemid','event'])

after = events_clean.count()
print('Avant nettoyage:', before)
print('Après nettoyage:', after)

events_clean.groupBy('event').count().show()


## 2) Pondération des interactions
view = 1, addtocart = 3, transaction = 5


In [ ]:
events_weighted = events_clean.withColumn(
    'weight',
    when(col('event')=='view', 1)
    .when(col('event')=='addtocart', 3)
    .when(col('event')=='transaction', 5)
    .otherwise(1)
)

events_weighted.groupBy('event').count().show()

events_weighted.select('weight').describe().show()


## 3) Export processed-data
On exporte en parquet dans `data/processed/events_clean`.


In [ ]:
import os

out_path = 'data/processed/events_clean'
os.makedirs('data/processed', exist_ok=True)

events_weighted.write.mode('overwrite').parquet(out_path)
print('Export OK ->', out_path)


## 4) (Optionnel) Upload MinIO
Si l'environnement MinIO est configuré via variables d'environnement, on peut uploader le dossier processed-data.


In [ ]:
# Exemple : utiliser le script existant si besoin
# from src.ingestion.upload_to_minio import ...
print('À configurer selon votre MinIO (optionnel).')
